In [ ]:
import os
import pandas as pd

# Odczyt datasetu

In [ ]:
df = pd.read_csv(os.path.join("../data", "domy.csv"))

# Usuwanie kolumn

# Zamiana kilku danych tekstowych na numeryczne

In [ ]:
del_cols = ["Id", "Street", "Utilities", "Condition2", "PoolQC", "PoolArea", "LowQualFinSF", "3SsnPorch"]
df = df.copy().drop(del_cols, axis=1)

In [ ]:
df = df.copy()
df["LotFrontage"] = pd.to_numeric(df["LotFrontage"], errors="coerce")
df["MasVnrArea"] = pd.to_numeric(df["MasVnrArea"], errors="coerce")
df["GarageYrBlt"] = pd.to_numeric(df["GarageYrBlt"], errors="coerce")

# Usuwanie outlierów

In [ ]:
out_cols = ["LotArea", "GrLivArea", "Fireplaces", "HalfBath", "GarageCars"]
ranges = [100000, 5000, 2, 1, 3]

df = df.copy()
for i in range(len(out_cols)):
    df = df[df[out_cols[i]] <= ranges[i]]

# Podział danych na zbiór treningowy i testowy

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inicjalizacja modelu i reportera do treningów

In [ ]:
from Regression.training_reporter import TrainingReporter
from Regression.training_model import CustomXGBRegressorModel

model = CustomXGBRegressorModel()
reporter = TrainingReporter(model, X_train, X_test, y_train, y_test)

# Pierwszy trening

In [ ]:
reporter.train()

# Kroswalidacja

In [ ]:
reporter.run_cross_validation(cv=10)

# Grid search

In [ ]:
grid_params = reporter.run_grid_search(cv=5)

# Kroswalidacja po dostosowaniu hiperparametrów za pomocą Grid Search

In [ ]:
from Regression.training_model import build_model_from_grid_params

model_GS = build_model_from_grid_params(grid_params.best_params_)
reporter_GS = TrainingReporter(model_GS, X_train, X_test, y_train, y_test)
reporter_GS.run_cross_validation(cv=10)

# Randomized grid search

In [ ]:
random_grid = reporter.run_randomized_search(cv=5)

# Kroswalidacja po dostosowaniu hiperparametrów za pomocą Grid Search

In [ ]:
model_RGS = build_model_from_grid_params(random_grid.best_params_)
reporter_RGS = TrainingReporter(model_RGS, X_train, X_test, y_train, y_test)
reporter_RGS.run_cross_validation(cv=10)